# Thí nghiệm 5: YOLOv8s + Xử lý Mất Cân Bằng (cls weight + copy_paste)

**Mục tiêu:** Cải thiện hiệu năng phát hiện các class thiểu số (Glass, Paper, Metal) bằng cách xử lý mất cân bằng class **trực tiếp trong quá trình huấn luyện**, không can thiệp vào pipeline dữ liệu.

**Phân tích mất cân bằng hiện tại:**

| Class | Số annotations | Tỉ lệ |
|-------|---------------|-------|
| Plastic | 2,074 | 43.3% |
| Other | 1,414 | 29.6% |
| Metal | 550 | 11.5% |
| Paper | 492 | 10.3% |
| Glass | 254 | 5.3% |

→ Tỉ lệ Plastic:Glass = **8:1** — cần xử lý để model không bị bias.

**Các kỹ thuật xử lý mất cân bằng được áp dụng:**
1. **`cls=2.0`** — Tăng trọng số classification loss (mặc định 0.5 → 2.0): phạt nặng hơn khi model nhầm class, buộc model chú trọng phân biệt Glass/Paper/Metal.
2. **`copy_paste=0.3`** — YOLO tự cắt object từ ảnh này dán vào ảnh khác (30% xác suất mỗi batch): tăng tần suất xuất hiện object thiểu số mà không làm hỏng bounding box.
3. **`mixup=0.0`** — Tắt mixup (gây nhiễu nhãn cho class thiểu số).

**So sánh với Baseline:**

| Tham số | Baseline | Thí nghiệm này |
|---------|----------|-----------------|
| Model | YOLOv8n | **YOLOv8s** |
| Epochs | 50 | **100** |
| Class imbalance | Không xử lý | **cls=2.0 + copy_paste=0.3** |
| Cosine LR | Không | **Có** |
| Early stopping | patience=100 | **patience=50** |
| MixUp | Không | **Tắt (0.0)** |
| Mosaic | 1.0 | 1.0 |
| close_mosaic | 10 | 10 |

**Pipeline:** Clone repo → Tải data → Làm sạch → COCO→YOLO → Chia tập → Train → Đánh giá trên Test

## 1. Clone repo

In [ ]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

## 2. Tải dữ liệu từ Google Drive

In [ ]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Chỉ cần tạo đến thư mục data (thư mục raw sẽ tự sinh ra khi giải nén)
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip
!rm -rf /kaggle/working/waste-detection_project/.git

## 3. Cài đặt thư viện

In [ ]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

## 4. Xử lý dữ liệu: Làm sạch → Chuyển đổi COCO→YOLO → Chia tập

> **Lưu ý:** KHÔNG chạy `letterbox.py` hay `oversample.py`.
> - `letterbox` gây xung đột với padding nội bộ của YOLO → làm lệch tọa độ bbox.
> - `oversample` với offline augmentation chồng lên online aug của YOLO → nhiễu nhãn.
> - Thay vào đó, mất cân bằng được xử lý bằng `cls weight` và `copy_paste` ngay trong `model.train()`.
>
> **`Training_dataYolo.py` tự ghi đường dẫn tuyệt đối vào `dataset.yaml`** — không cần sửa thủ công.

In [ ]:
# Làm sạch dữ liệu
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

# Chuyển COCO sang YOLO format (tự sinh dataset.yaml với đường dẫn tuyệt đối)
%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

# Chia tập train/val/test (70/15/15)
%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

## 5. Huấn luyện YOLOv8s — Xử lý mất cân bằng class

**Giải thích các tham số xử lý mất cân bằng:**

- **`cls=2.0`**: Tăng trọng số của classification loss lên 4x so với mặc định (0.5). Model sẽ bị phạt nặng hơn khi nhầm class → buộc phải học phân biệt Glass/Paper/Metal tốt hơn.
- **`copy_paste=0.3`**: Với xác suất 30%, YOLO sẽ cắt object từ một ảnh khác trong batch và dán vào ảnh hiện tại kèm bounding box. Vì Glass/Paper/Metal ít xuất hiện hơn, chúng sẽ được "paste" nhiều hơn tương đối → model thấy class thiểu số thường xuyên hơn.
- **`copy_paste_mode="flip"`**: Flip object trước khi paste để tăng đa dạng góc nhìn mà không làm biến dạng hình học.
- **`mixup=0.0`**: Tắt hoàn toàn MixUp. MixUp trộn 2 ảnh + nhãn → class thiểu số bị "pha loãng" trong ảnh hỗn hợp → phản tác dụng với mục tiêu xử lý mất cân bằng.
- **`mosaic=1.0`**: Giữ nguyên Mosaic (ghép 4 ảnh). Mosaic giúp model thấy nhiều context khác nhau và **kết hợp tốt với copy_paste** để tăng density object thiểu số.

In [ ]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Quay về thư mục experimental trước khi train
%cd /kaggle/working/waste-detection_project/notebooks/experimental

BASE_DIR = Path('/kaggle/working/waste-detection_project')
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("=" * 60)
print("  THI NGHIEM 5: YOLOv8s + cls_weight + copy_paste")
print("  Xu ly mat can bang class khong can oversample thu cong")
print("=" * 60)

model_path = BASE_DIR / "models/yolov8s.pt"
model = YOLO(str(model_path))

train_results = model.train(
    data         = str(YAML_PATH),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    name         = "exp5_yolov8s_imbalance",
    project      = str(RESULTS / "runs"),
    exist_ok     = True,
    verbose      = True,

    # --- Tham số huấn luyện ---
    patience     = 50,          # Nới lỏng patience khi dùng cos_lr (cos_lr có plateau tự nhiên)
    cos_lr       = True,        # Cosine Annealing LR — giảm mượt hơn step LR
    close_mosaic = 10,          # Tắt mosaic 10 epoch cuối để fine-tune

    # --- Xử lý mất cân bằng class ---
    cls          = 2.0,         # Tăng classification loss weight (default=0.5)
                                # Phạt nặng hơn khi nhầm class → model tập trung vào Glass/Metal/Paper

    # --- Augmentation: dùng copy_paste thay oversample ---
    copy_paste      = 0.3,      # 30% xác suất paste thêm object từ ảnh khác
    copy_paste_mode = "flip",   # Flip object trước khi paste (tăng đa dạng, tránh artifact)
    mosaic          = 1.0,      # Giữ mosaic mạnh — kết hợp tốt với copy_paste
    mixup           = 0.0,      # TẮT mixup — gây nhiễu nhãn cho class thiểu số

    # --- Augmentation cơ bản (giữ default YOLO) ---
    fliplr       = 0.5,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    scale        = 0.5,
    translate    = 0.1,
)

print("\n" + "=" * 60)
print("  HUAN LUYEN HOAN TAT!")
print("=" * 60)

## 6. Đánh giá trên tập Test

In [ ]:
# Đánh giá trên tập Test (dùng best.pt)
print("\n[EVAL] Danh gia tren tap Test...")
metrics = model.val(split="test", verbose=True)

map50   = float(metrics.box.map50)
map5095 = float(metrics.box.map)
prec    = float(metrics.box.mp)
recall  = float(metrics.box.mr)

print(f"\n{'='*45}")
print(f"  KET QUA TREN TAP TEST")
print(f"{'='*45}")
print(f"  mAP@0.5     : {map50:.4f}  ({map50*100:.2f}%)")
print(f"  mAP@0.5:0.95: {map5095:.4f}  ({map5095*100:.2f}%)")
print(f"  Precision   : {prec:.4f}  ({prec*100:.2f}%)")
print(f"  Recall      : {recall:.4f}  ({recall*100:.2f}%)")
print(f"{'='*45}")

# Đo tốc độ inference
val_imgs = glob.glob(
    str(BASE_DIR / "data/processed/images/test/**/*.*"), recursive=True
)[:50]

if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"  Inference   : {elapsed:.1f} ms/anh")
else:
    elapsed = None
    print("  [WARN] Khong tim thay anh test de do inference.")

## 7. Lưu kết quả & Xuất file ZIP

In [ ]:
# Lưu kết quả vào CSV
result_row = {
    "Model": "YOLOv8s (100ep, cls=2.0, copy_paste=0.3)",
    "mAP@0.5 (%)": round(map50 * 100, 2),
    "mAP@0.5:0.95 (%)": round(map5095 * 100, 2),
    "Precision (%)": round(prec * 100, 2),
    "Recall (%)": round(recall * 100, 2),
    "Inference (ms)": round(elapsed, 1) if elapsed else "N/A",
}

df = pd.DataFrame([result_row])
csv_path = RESULTS / "ket_qua_exp5_imbalance.csv"
df.to_csv(csv_path, index=False)

print(f"\nDa luu ket qua tai: {csv_path}")
print(df.to_string(index=False))

In [ ]:
# Nén kết quả để tải về
!zip -r /kaggle/working/exp5_yolov8s_imbalance_results.zip /kaggle/working/waste-detection_project/results/runs/exp5_yolov8s_imbalance
print("\nFile ZIP da san sang de tai ve!")